# Analysis: Experiment \#A001 - Checking pdf metadata

**Technical summary**
| Exp Nr | Responsible | Date | Description |
| :----: | :---------- | :--: | :---------- |
| A001 | Mario Tormo Romero | 2026-03-03 | Analyzing metadata in pdf files |

<br>

| Results |
| :------ |
| Short text with the results of the experiment |

---
## Objective
In this experiment we aim to determine the amount and quality of the metadata contained in pdf files in different time periods, and thus, with different formats.

---
## Setting
In this experiment we use the following libraries/tools:
- [PyPDF2](https://pypdf2.readthedocs.io/) for extracting the metadata from the pdf files
- [pandas](https://pandas.pydata.org/) for analyzing the metadata

---
## Experiment evaluation
Here the results

---
## Conclusion
What have we learnt from this experiment

---
## Links
Here a list with links to objects, logs and reports

---
## Dev

### 1. Setting up
Preparing libraries and constants.

In [11]:
from pathlib import Path
from PyPDF2 import PdfReader

import pandas as pd

from datetime import datetime


DATA_PATH = Path('../03_data/')


### 2. Extracting metadata from a file
We define now a function for extracting the metadata from a specific file.

In [3]:
def extract_pdf_metadata(pdf_path: Path) -> dict:
    """Extract metadata from a single PDF file."""
    try:
        reader = PdfReader(pdf_path)
        metadata = reader.metadata or {}

        # Convert metadata keys to readable strings
        clean_metadata = {
            key.lstrip("/"): value
            for key, value in metadata.items()
        }

        return clean_metadata

    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return {}
    
# Test for the function
file_path = DATA_PATH / "Ausarbeitungen/WD 1-019-24; WD 7-060-24.pdf"
metadata_dict = extract_pdf_metadata(file_path)
metadata_dict

{'Subject': '',
 'Author': 'Wissenschaftliche Dienste des Deutschen Bundestages',
 'Keywords': '',
 'CreationDate': "D:20241017112032+02'00'",
 'ModDate': "D:20241017112032+02'00'",
 'Title': 'Sachstand - Die Lootbox im deutschen Recht und im europäischen Vergleich'}

### 3. Converting dates to datetime format
We use a function for turning strings with dates into datetime objects

In [4]:
def convert_date(date_str: str) -> datetime:
    """
    Convert a string containing pdf date info (e.g., D:20241017112032+02'00') into a datetime object.
    """
    try:
        match = re.match(r"(D:\d{4})(\d{2})?(\d{2})?(\d{2})?(\d{2})?(\d{2})?([+\-Z])?(\d{2})?'?(\d{2})?'?", date_str)

        if not (date_str.startswith("D:") and match and len(date_str)==23):
            raise ValueError(f"Invalid PDF date format: {date_str}")
    
        date_str = date_str[2:].replace("'", "")  # Remove the "D:" prefix and the single quotation marks
        dt = datetime.strptime(date_str, "%Y%m%d%H%M%S%z") # Turn the date into datetime object

        return dt
    except Exception as e:
        raise ValueError(f"Invalid PDF date format: {date_str}") from e
        

# Test for the function
pdf_date_str = "D:20241017112031+02'00'"
dt_obj = convert_date(pdf_date_str)
print(dt_obj)

2024-10-17 11:20:31+02:00


### 4. Updating our original function
We add a call to the new function in the original <code>extract_pdf_metadata</code> function

In [5]:

def extract_pdf_metadata(pdf_path: Path) -> dict:
    """
    Extract metadata from a single PDF file and clean it by stripping slashes from the keys and turning strings
    containing dates into datetime objects.
    """
    date_keys = ['CreationDate', 'ModDate']
    
    try:
        reader = PdfReader(pdf_path)
        metadata = reader.metadata or {}

        # Convert metadata keys to readable strings
        clean_metadata = {
            key.lstrip("/"): value
            for key, value in metadata.items()
        }
        
        for key in date_keys:
            clean_metadata[key] = convert_date(clean_metadata[key])

        return clean_metadata

    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return {}
    
# Test for the function
file_path = DATA_PATH / "Ausarbeitungen/WD 1-019-24; WD 7-060-24.pdf"
metadata_dict = extract_pdf_metadata(file_path)
metadata_dict

{'Subject': '',
 'Author': 'Wissenschaftliche Dienste des Deutschen Bundestages',
 'Keywords': '',
 'CreationDate': datetime.datetime(2024, 10, 17, 11, 20, 32, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
 'ModDate': datetime.datetime(2024, 10, 17, 11, 20, 32, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
 'Title': 'Sachstand - Die Lootbox im deutschen Recht und im europäischen Vergleich'}

### 5. Parsing all pdfs at the same time
We create functions for parsing several pdfs at the same time

In [8]:

def collect_pdf_filenames(file_path):
    # Get all PDF files recursively
    pdf_files = list(Path(file_path).glob("**/*.pdf"))

    return pdf_files

# Test the function
pdf_list = collect_pdf_filenames(DATA_PATH / 'Ausarbeitungen')
pdf_list

[PosixPath('../03_data/Ausarbeitungen/WD 10-013-23.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 1-019-24; WD 7-060-24.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 2-029-25.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 6-094-23.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 8-011-22.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 10-042-22.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 9-100-21.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 4-086-24.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 2-027-25_EN.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 6-052-24.pdf'),
 PosixPath('../03_data/Ausarbeitungen/EU 6-012-25.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 3-029-23.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 8-013-22.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 7-085-22; WD 5-124-22.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 5-009-25.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 9-068-23.pdf'),
 PosixPath('../03_data/Ausarbeitungen/WD 7-051-24.pdf')]

In [10]:

def extract_metadata_from_folder(folder_path: Path):
    """Recursively extract metadata from all PDF files in a folder."""
    metadata_dict = {}
    pdf_list = collect_pdf_filenames(folder_path)
    
    for pdf_file in pdf_list:
        metadata_dict[pdf_file.name] = {'path': pdf_file,
                                        'metadata': extract_pdf_metadata(pdf_file)}
    
    return metadata_dict

# Test function
metadata_dict = extract_metadata_from_folder(DATA_PATH / 'Ausarbeitungen')
metadata_dict

{'WD 10-013-23.pdf': {'path': PosixPath('../03_data/Ausarbeitungen/WD 10-013-23.pdf'),
  'metadata': {'Author': 'Wissenschaftliche Dienste des Deutschen Bundestages',
   'Keywords': '',
   'Producer': 'axesWord',
   'Creator': 'Microsoft® Word 2013 (15.0.5571)',
   'Subject': '',
   'Title': 'Spitzensportförderung in ausgewählten europäischen Ländern',
   'CreationDate': datetime.datetime(2023, 7, 25, 12, 3, 53, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
   'ModDate': datetime.datetime(2023, 7, 25, 12, 3, 53, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200)))}},
 'WD 1-019-24; WD 7-060-24.pdf': {'path': PosixPath('../03_data/Ausarbeitungen/WD 1-019-24; WD 7-060-24.pdf'),
  'metadata': {'Subject': '',
   'Author': 'Wissenschaftliche Dienste des Deutschen Bundestages',
   'Keywords': '',
   'CreationDate': datetime.datetime(2024, 10, 17, 11, 20, 32, tzinfo=datetime.timezone(datetime.timedelta(seconds=7200))),
   'ModDate': datetime.datetime(2024, 10, 17, 11, 20

### 6. Analyzing the data with pandas
We dump the data into a pandas dataframe for analyzing it.

In [14]:
df = pd.DataFrame.from_dict(
    {k: v["metadata"] for k, v in metadata_dict.items()},
    orient="index"
)
df


,Author,Keywords,Producer,Creator,Subject,Title,CreationDate,ModDate
WD 10-013-23.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,axesWord,Microsoft® Word 2013 (15.0.5571),,Spitzensportförderung in ausgewählten europäis...,2023-07-25 12:03:53+02:00,2023-07-25 12:03:53+02:00
WD 1-019-24; WD 7-060-24.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,NaN,NaN,,Sachstand - Die Lootbox im deutschen Recht und...,2024-10-17 11:20:32+02:00,2024-10-17 11:20:32+02:00
WD 2-029-25.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,NaN,NaN,,Die israelische Militäroperation Rising Lion u...,2025-07-04 13:46:12+02:00,2025-07-07 07:11:04+02:00
WD 6-094-23.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,NaN,NaN,,Einzelfrage zu Leistungen in besonderen Fällen...,2023-12-07 10:55:08+01:00,2024-01-19 10:06:08+01:00
WD 8-011-22.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,axesWord,Microsoft® Word 2013,,Dokumentation,2022-03-03 14:43:34+01:00,2022-03-03 14:43:34+01:00
WD 10-042-22.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,axesWord,Microsoft® Word 2013 (15.0.5529),,Zum Einsatz von Open-Source-Software in EU-Mit...,2023-03-13 14:46:17+01:00,2023-03-13 14:46:17+01:00
WD 4-086-24.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,NaN,NaN,,Zur Frage der Verfassungsmäßigkeit einer Übers...,2025-01-24 11:11:56+01:00,2025-01-24 11:11:56+01:00
WD 2-027-25_EN.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,NaN,NaN,,Regulations of state legal representation in i...,2025-07-28 14:57:26+02:00,2025-07-28 14:57:26+02:00
WD 6-052-24.pdf,Wissenschaftliche Dienste des Deutschen Bundes...,,NaN,NaN,,Aktuelle Informationen zur betrieblichen Alter...,2024-08-21 10:31:00+02:00,2024-08-21 10:31:00+02:00
EU 6-012-25.pdf,Unterabteilung Europa - Fachbereich Europa,,NaN,NaN,,Familiennachzug von subsidiär Schutzberechtigt...,2025-03-19 10:15:20+01:00,2025-03-19 10:15:20+01:00
